# 03 · Exploratory Data Analysis of the table corpus

Visual + statistical sanity checks **before** burning hours of training:
image grid, structural difficulty distribution, target-length distribution
(drives `max_seq_len`), and pixel statistics (drives normalization).

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from PIL import Image

records = [json.loads(l) for l in open(cfg.paths.manifest_train)]
print(len(records), "train records")

## 1. Visual grid of random samples

In [ ]:
import random
random.seed(0)
picks = random.sample(records, 9)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax, rec in zip(axes.flat, picks):
    ax.imshow(Image.open(rec["image"])); ax.axis("off")
    obj = json.loads(rec["json"])
    ax.set_title(f"{len(obj['rows'])}r x {len(obj['columns'])}c"
                 + (" +groups" if "column_groups" in obj else ""), fontsize=9)
plt.suptitle("random training tables"); plt.tight_layout(); plt.show()

## 2. Structural difficulty distribution

In [ ]:
rows = [len(json.loads(r["json"])["rows"]) for r in records]
cols = [len(json.loads(r["json"])["columns"]) for r in records]
grouped = [1 if "column_groups" in json.loads(r["json"]) else 0 for r in records]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(rows, bins=range(min(rows), max(rows)+2)); axes[0].set_title("rows / table")
axes[1].hist(cols, bins=range(min(cols), max(cols)+2)); axes[1].set_title("cols / table")
axes[2].bar(["plain", "merged headers"],
            [len(grouped)-sum(grouped), sum(grouped)]); axes[2].set_title("header complexity")
plt.tight_layout(); plt.show()

## 3. Target token lengths → validates `model.max_seq_len`

In [ ]:
from gemma_ft_json.models import load_gemma_local
from gemma_ft_json.utils import resolve_dtype
_, tok = load_gemma_local(cfg.paths.gemma_model_dir, resolve_dtype(cfg.device.dtype))

lens = [len(tok.encode(r["json"], add_special_tokens=False)) for r in records[:500]]
plt.figure(figsize=(7, 3.5))
plt.hist(lens, bins=40)
plt.axvline(cfg.model.max_seq_len, color="red", ls="--",
            label=f"max_seq_len={cfg.model.max_seq_len}")
plt.title("JSON target length (tokens)"); plt.legend(); plt.show()
print(f"p50={np.percentile(lens,50):.0f}  p95={np.percentile(lens,95):.0f}  "
      f"max={max(lens)}  → truncated: {sum(l > cfg.model.max_seq_len for l in lens)}")

## 4. Pixel statistics → validates the document normalization (0.9 / 0.2)

In [ ]:
arrs = [np.asarray(Image.open(r["image"]).convert("L"), dtype=np.float32)/255.
        for r in random.sample(records, 50)]
flat = np.concatenate([a.ravel() for a in arrs])
plt.figure(figsize=(7, 3))
plt.hist(flat, bins=60); plt.title("grayscale pixel distribution (50 images)")
plt.show()
print(f"mean={flat.mean():.3f}  std={flat.std():.3f}  (dataset.py uses 0.9 / 0.2)")